In [12]:
!pip install pandas scikit-learn

In [13]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
from google.colab import files

uploaded = files.upload()

Saving coursera_courses.csv to coursera_courses.csv


In [19]:
df = pd.read_csv("coursera_courses.csv")

df.head()

,Unnamed: 0,Title,Organization,Skills,Ratings,Review counts,Metadata
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,4.8(20K reviews),Beginner · Professional Certificate · 3 - 6 Mo...
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,4.8(137K reviews),Beginner · Professional Certificate · 3 - 6 Mo...
2,2,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,4.8(100K reviews),Beginner · Professional Certificate · 3 - 6 Mo...
3,3,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,4.6(120K reviews),Beginner · Professional Certificate · 3 - 6 Mo...
4,4,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,4.8(23K reviews),Beginner · Professional Certificate · 3 - 6 Mo...


In [20]:
df = pd.read_csv("coursera_courses.csv")

df.head()

,Unnamed: 0,Title,Organization,Skills,Ratings,Review counts,Metadata
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,4.8(20K reviews),Beginner · Professional Certificate · 3 - 6 Mo...
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,4.8(137K reviews),Beginner · Professional Certificate · 3 - 6 Mo...
2,2,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,4.8(100K reviews),Beginner · Professional Certificate · 3 - 6 Mo...
3,3,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,4.6(120K reviews),Beginner · Professional Certificate · 3 - 6 Mo...
4,4,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,4.8(23K reviews),Beginner · Professional Certificate · 3 - 6 Mo...


In [22]:
print(df.columns)

Index(['Unnamed: 0', 'Title', 'Organization', 'Skills', 'Ratings',
       'Review counts', 'Metadata'],
      dtype='object')


In [25]:
df = df.drop(columns=["Unnamed: 0"])

In [26]:
df.isnull().sum()

,0
Title,0
Organization,0
Skills,0
Ratings,0
Review counts,0
Metadata,0


In [27]:
df.fillna("", inplace=True)

In [28]:
df["combined_features"] = (
    df["Title"] + " " +
    df["Organization"] + " " +
    df["Skills"] + " " +
    df["Metadata"]
)

In [29]:
df[["Title", "combined_features"]].head()

,Title,combined_features
0,Google Cybersecurity,"Google Cybersecurity Google Network Security,..."
1,Google Data Analytics,"Google Data Analytics Google Data Analysis, R..."
2,Google Project Management:,Google Project Management: Google Project Man...
3,IBM Data Science,"IBM Data Science IBM Python Programming, Data..."
4,Google Digital Marketing & E-commerce,Google Digital Marketing & E-commerce Google ...


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(df["combined_features"])

print(tfidf_matrix.shape)

(623, 1039)


In [31]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_courses(user_interest, top_n=5):

    # Convert user input into TF-IDF vector
    user_vector = tfidf.transform([user_interest])

    # Calculate cosine similarity
    similarity = cosine_similarity(user_vector, tfidf_matrix)

    similarity_scores = similarity.flatten()

    # Get indices of top recommendations
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    # Select recommendations
    recommendations = df.iloc[top_indices].copy()

    recommendations["Similarity Score"] = similarity_scores[top_indices]

    return recommendations[
        ["Title",
         "Organization",
         "Ratings",
         "Review counts",
         "Similarity Score"]
    ]

In [32]:
recommend_courses("python machine learning")

,Title,Organization,Ratings,Review counts,Similarity Score
258,Applied Machine Learning in Python,University of Michigan,4.6,4.6(8.4K reviews),0.771947
7,Machine Learning,Multiple educators,4.9,4.9(19K reviews),0.736517
46,Supervised Machine Learning: Regression and Cl...,DeepLearning.AI,4.9,4.9(16K reviews),0.722802
87,Advanced Learning Algorithms,DeepLearning.AI,4.9,4.9(4.3K reviews),0.686940
415,MLOps | Machine Learning Operations,Duke University,4.0,4.0(150 reviews),0.665426


In [33]:
while True:

    user_interest = input("\nEnter your interests (type 'exit' to quit): ")

    if user_interest.lower() == "exit":
        print("Goodbye!")
        break

    results = recommend_courses(user_interest)

    print("\nTop Recommended Courses:\n")

    for i, row in results.iterrows():

        print("="*60)
        print("Course        :", row["Title"])
        print("Organization  :", row["Organization"])
        print("Rating        :", row["Ratings"])
        print("Reviews       :", row["Review counts"])
        print("Similarity    :", round(row["Similarity Score"]*100,2), "%")


Enter your interests (type 'exit' to quit): java

Top Recommended Courses:

Course        : Introduction to Programming with Python and Java
Organization  : University of Pennsylvania
Rating        : 4.4
Reviews       : 4.4(1.5K reviews)
Similarity    : 39.96 %
Course        : Object Oriented Programming in Java
Organization  : Multiple educators
Rating        : 4.6
Reviews       : 4.6(15K reviews)
Similarity    : 39.58 %
Course        : JavaScript Security
Organization  : Infosec
Rating        : 4.5
Reviews       : 4.5(66 reviews)
Similarity    : 38.03 %
Course        : Building Scalable Java Microservices with Spring Boot and Spring Cloud
Organization  : Google Cloud
Rating        : 4.3
Reviews       : 4.3(1.2K reviews)
Similarity    : 37.42 %
Course        : Java Programming and Software Engineering Fundamentals
Organization  : Duke University
Rating        : 4.6
Reviews       : 4.6(22K reviews)
Similarity    : 33.11 %

Enter your interests (type 'exit' to quit): python

Top Recomm